# Iris Classification with Neural Networks

In [60]:
import pandas as pd

from keras import layers, backend, Sequential
from scikeras.wrappers import KerasClassifier
from sklearn.model_selection import GridSearchCV, KFold, cross_val_score

from functools import partial

## Dataset

In [61]:
df = pd.read_csv("datasets/iris.csv")
display(df.head())
display(df.describe())

,sepal length,sepal width,petal length,petal width,class
0,5.1,3.5,1.4,0.2,Iris-setosa
1,4.9,3.0,1.4,0.2,Iris-setosa
2,4.7,3.2,1.3,0.2,Iris-setosa
3,4.6,3.1,1.5,0.2,Iris-setosa
4,5.0,3.6,1.4,0.2,Iris-setosa


,sepal length,sepal width,petal length,petal width
count,150.000000,150.000000,150.000000,150.000000
mean,5.843333,3.054000,3.758667,1.198667
std,0.828066,0.433594,1.764420,0.763161
min,4.300000,2.000000,1.000000,0.100000
25%,5.100000,2.800000,1.600000,0.300000
50%,5.800000,3.000000,4.350000,1.300000
75%,6.400000,3.300000,5.100000,1.800000
max,7.900000,4.400000,6.900000,2.500000


### Encode

In [62]:
df_encoded = pd.get_dummies(df, columns=["class"], prefix="class", dtype=int)
display(df_encoded.sample(5))

,sepal length,sepal width,petal length,petal width,class_Iris-setosa,class_Iris-versicolor,class_Iris-virginica
115,6.4,3.2,5.3,2.3,0,0,1
130,7.4,2.8,6.1,1.9,0,0,1
21,5.1,3.7,1.5,0.4,1,0,0
84,5.4,3.0,4.5,1.5,0,1,0
107,7.3,2.9,6.3,1.8,0,0,1


### Feature/Target Split

In [63]:
X = df_encoded[["sepal length", "sepal width", "petal length", "petal width"]]
y = df_encoded[["class_Iris-setosa", "class_Iris-versicolor", "class_Iris-virginica"]]

display(X)
display(y)

,sepal length,sepal width,petal length,petal width
0,5.1,3.5,1.4,0.2
1,4.9,3.0,1.4,0.2
2,4.7,3.2,1.3,0.2
3,4.6,3.1,1.5,0.2
4,5.0,3.6,1.4,0.2
...,...,...,...,...
145,6.7,3.0,5.2,2.3
146,6.3,2.5,5.0,1.9
147,6.5,3.0,5.2,2.0
148,6.2,3.4,5.4,2.3


,class_Iris-setosa,class_Iris-versicolor,class_Iris-virginica
0,1,0,0
1,1,0,0
2,1,0,0
3,1,0,0
4,1,0,0
...,...,...,...
145,0,0,1
146,0,0,1
147,0,0,1
148,0,0,1


## Model

### Definition

In [64]:
def get_neural_network(
    hidden_layers:int,
    activation:str,
    kernel_initializer:str,
    optimizer:str,
) -> Sequential:
    backend.clear_session()


    model = Sequential()
    model.add(layers.InputLayer(shape=(4,)))

    for _ in range(hidden_layers):
        model.add(layers.Dense(
            units=4, 
            activation=activation,
            kernel_initializer=kernel_initializer
        ))

    model.add(layers.Dense(units=3, activation="softmax"))


    model.compile(
        optimizer=optimizer,
        loss="categorical_crossentropy",
        metrics=['categorical_accuracy']
    )
    
    return model

model = KerasClassifier(model=get_neural_network, verbose=0)

### Tuning

In [ ]:
params = {
    'batch_size': [10, 20],
    'epochs': [50, 100],
    'model__hidden_layers': [0, 1, 2],
    'model__activation': ['relu', 'tanh'],
    'model__kernel_initializer': ['normal', 'random_uniform'],
    'model__optimizer': ['adam', 'sgd']
}

tuning_results = GridSearchCV(
    estimator=model,
    param_grid=params,
    scoring='accuracy',
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    verbose=2
).fit(X, y) # 32 min

In [ ]:
print(tuning_results.best_params_) # {'batch_size': 10, 'epochs': 100, 'model__activation': 'tanh', 'model__hidden_layers': 1, 'model__kernel_initializer': 'normal', 'model__optimizer': 'adam'}
print(tuning_results.best_score_) # 0.9800000000000001

### Best Model Evaluation

In [65]:
best_model = KerasClassifier(
    model = partial(
        get_neural_network,
        hidden_layers=1,
        activation='tanh',
        kernel_initializer='normal',
        optimizer='adam',
    ),
    epochs=100,
    batch_size=10,
    verbose=0
)

results = cross_val_score(
    estimator=best_model,
    X=X, y=y,
    cv=KFold(n_splits=5, shuffle=True, random_state=42),
    scoring='accuracy',
)

In [66]:
print(f"Accuracies: {results}")
print(f"Mean: {results.mean()}, Std: {results.std()}")

Accuracies: [0.96666667 0.96666667 0.93333333 0.93333333 1.        ]
Mean: 0.96, Std: 0.024944382578492935


## Final Model

In [ ]:
classifier = KerasClassifier(
    model = partial(
        get_neural_network,
        hidden_layers=2,
        activation='tanh',
        kernel_initializer='normal',
        optimizer='adam',
    ),
    epochs=100,
    batch_size=10,
    verbose=0
).fit(
    X=X,
    y=y
)

In [ ]:
classifier.model_.summary()
classifier.model_.save('models/classifier.keras')